# Notebook 02 — PyTorch Tensors, Autograd, Modules, Data, and Training

    ## Learning objectives

    - Reason precisely about tensor shape, storage, dtype, and device
- Use autograd and nn.Module without hidden state mistakes
- Construct, evaluate, checkpoint, and debug a complete training pipeline

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from the Colab Secrets UI without displaying it. Create a
# secret named exactly HF_TOKEN and enable notebook access with its toggle.
token = os.getenv("HF_TOKEN")
token_error = None
if IN_COLAB and not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        token_error = type(exc).__name__
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub environment variable.
if token:
    os.environ["HF_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("HF_TOKEN is unavailable. In Colab, open the key icon (Secrets), add HF_TOKEN, ")
    print("enable its Notebook access toggle, and rerun this cell. Public models still work.")
    if token_error:
        print("Colab secret lookup status:", token_error)


## 2.1 Tensor construction, dtype, and device

A tensor combines typed storage with shape, strides, device, and optional gradient history. Constructors differ: `tensor` copies data and infers a dtype, `as_tensor` may share compatible storage, and factory methods such as `zeros_like` inherit properties. Token IDs and class labels are normally `long`; activations and parameters are floating point; masks are boolean. Avoid changing the global default dtype as a convenience because it can silently alter modules and constants. Select CUDA, Apple MPS, or CPU explicitly, construct modules before moving them, and transfer each batch at the training boundary. Operations generally require compatible devices.


In [ ]:
import torch

def default_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

device=default_device()
ids=torch.tensor([[1,2,3],[4,5,0]],dtype=torch.long)
mask=ids.ne(0)
x=torch.arange(12,dtype=torch.float32).reshape(2,2,3)
print(device,ids.dtype,mask.dtype,x.shape,x.device)


## 2.2 Shapes, indexing, broadcasting, and contraction

LLM tensors repeatedly use batch, time, head, and feature axes. Basic slicing returns a view when possible; integer indexing removes an axis; a length-one slice preserves it; advanced indexing and boolean selection often allocate copies. Broadcasting aligns dimensions from the right and does not itself copy expanded values, but a later operation may materialize a large result. Use `unsqueeze`, `expand`, and named shape comments deliberately. Matrix multiplication interprets the final two dimensions specially, while `einsum` expresses arbitrary contractions. Assert every boundary and compare a compact vectorized expression with a loop on tiny data.


In [ ]:
torch.manual_seed(2)
B,T,H,D=2,3,2,4
q=torch.randn(B,H,T,D); k=torch.randn(B,H,T,D)
scores=q@k.transpose(-2,-1) / D**0.5
reference=torch.empty_like(scores)
for b in range(B):
 for h in range(H):
  reference[b,h]=q[b,h]@k[b,h].T / D**0.5
print(scores.shape,torch.testing.assert_close(scores,reference))
bias=torch.arange(T).view(1,1,1,T); print((scores+bias).shape)


## 2.3 Storage, views, copies, and mutation

A transpose typically changes strides without rearranging storage, so the result is non-contiguous. `view` requires compatible strides, while `reshape` may return a view or silently copy. `contiguous` materializes the current logical order. `clone` copies while preserving an autograd relationship; `detach` shares storage but removes the gradient edge; `detach().clone()` creates independent history-free data. In-place operations carry a trailing underscore and can invalidate saved tensors needed for backward. Never use `.data` to bypass autograd. Understanding aliasing matters for activation memory, parameter tying, cached tensors, and bugs where one edit unexpectedly changes another value.


In [ ]:
base=torch.arange(24.).reshape(2,3,4)
transposed=base.transpose(1,2)
print(base.stride(),transposed.stride(),transposed.is_contiguous())
flat=transposed.contiguous().view(2,-1)
alias=base[:,0]; copied=base[:,0].clone()
base[0,0,0]=-99
print(flat.shape,alias[0,0].item(),copied[0,0].item(),transposed.untyped_storage().data_ptr()==base.untyped_storage().data_ptr())


## 2.4 Autograd as vector–Jacobian products

PyTorch records differentiable operations when gradient mode is enabled and at least one input requires gradients. Calling backward on a scalar seeds its derivative with one; a non-scalar output needs an explicit upstream vector. Reverse mode is ideal when one loss depends on many parameters. Gradients accumulate into leaf `.grad`, so accumulation can be intentional across microbatches but must be cleared at update boundaries. Intermediate gradients are normally discarded unless retained. A completed backward normally frees its graph, so a distinct calculation should run a new forward pass. `no_grad` suppresses recording and `inference_mode` adds stronger evaluation optimizations. Detaching, converting to NumPy, constructing a new tensor from values, or branching through nondifferentiable logic can break the graph.


In [ ]:
w=torch.tensor([2.,-1.],requires_grad=True); features=torch.tensor([[3.,4.],[1.,2.]])
y=features@w
y.backward(torch.tensor([1.,.5]))
print("vector-Jacobian product",w.grad)
w.grad=None
y=features@w; y.retain_grad()  # a fresh graph; retain this non-leaf gradient for inspection
loss=y.square().mean(); loss.backward(); print("scalar loss gradient",w.grad,"leaf",w.is_leaf,"retained y grad",y.grad)


## 2.5 Gradient validation and diagnostics

Autograd differentiates the implemented program, not the program you intended. Validate unfamiliar objectives or custom operations with central finite differences or `gradcheck` in float64 on a tiny input away from nondifferentiable points. Compare multiple step sizes because truncation and rounding errors compete. During real training, inspect missing and non-finite gradients, global norm, parameter-to-update ratios, and selected layer norms. Hooks can observe gradients but must be removed and should not mutate values casually. An absent gradient usually indicates a detach, unused parameter, optimizer omission, or conditional path—not a reason to add `retain_graph=True`.


In [ ]:
def objective(z): return (z.sin()*z.square()).sum()
z=torch.tensor([.4,-.7],dtype=torch.float64,requires_grad=True)
objective(z).backward(); analytic=z.grad.detach().clone(); eps=1e-6
numeric=[]
for i in range(z.numel()):
 plus=z.detach().clone(); minus=z.detach().clone(); plus[i]+=eps; minus[i]-=eps
 numeric.append(((objective(plus)-objective(minus))/(2*eps)).item())
print(analytic.tolist(),numeric); torch.testing.assert_close(analytic,torch.tensor(numeric,dtype=z.dtype),rtol=1e-5,atol=1e-7)


## 2.6 Modules, parameters, buffers, initialization, and modes

`nn.Module` registers assigned child modules and `Parameter` objects recursively. This powers device moves, mode changes, state dictionaries, and parameter discovery by optimizers. Plain tensor attributes are not parameters; persistent non-trainable state such as positional indices belongs in a registered buffer. Create layers in `__init__` rather than inside `forward`, where new random parameters would appear on every call. `train()` and `eval()` change modules such as dropout and normalization but do not enable or disable gradients. Initialize deliberately, inspect named parameters and buffers, count unique trainable values, and remember that tied parameters may appear through multiple logical paths while sharing storage.


In [ ]:
from torch import nn
class TinyClassifier(nn.Module):
 def __init__(self,vocab=16,width=8,classes=3):
  super().__init__(); self.embed=nn.Embedding(vocab,width,padding_idx=0); self.norm=nn.LayerNorm(width); self.head=nn.Linear(width,classes); self.register_buffer("version",torch.tensor(1))
 def forward(self,ids,mask):
  h=self.norm(self.embed(ids)); pooled=(h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1); return self.head(pooled)
model=TinyClassifier(); print(sum(p.numel() for p in model.parameters()),list(dict(model.named_buffers())),list(model.state_dict()))


## 2.7 Objectives, masks, and reductions

Cross-entropy consumes unnormalized logits and integer class IDs; applying softmax first is numerically and mathematically wrong. For language modeling, flatten vocabulary logits and labels only after aligning the causal shift. Padding or prompt positions use an ignore index or an explicit mask. Reduction determines statistical weighting: averaging per-batch means biases results when valid-token counts differ, whereas summing eligible losses and dividing by the total eligible count is token weighted. Attention masks control which representations interact; loss masks control which targets contribute. They are related but not interchangeable. Assert at least one valid target and test a hand-computed example.


In [ ]:
logits=torch.tensor([[[2.,0.,-1.],[0.,1.,2.]],[[1.,1.,1.],[4.,0.,0.]]])
labels=torch.tensor([[0,2],[1,-100]])
per_token=nn.functional.cross_entropy(logits.flatten(0,1),labels.flatten(),ignore_index=-100,reduction="none").view_as(labels)
valid=labels.ne(-100); loss=per_token.sum()/valid.sum()
print(per_token,valid,loss.item())
manual=-torch.log_softmax(logits,-1)[valid,labels[valid]].mean(); torch.testing.assert_close(loss,manual)


## 2.8 Dataset, sampler, DataLoader, and collator

A map-style Dataset maps indices to records; an iterable dataset streams records and requires explicit worker sharding. A sampler determines order, the DataLoader fetches records, and `collate_fn` creates a rectangular batch. Variable-length token sequences need padding or packing plus masks and labels. The collator—not the model—should own padding policy. Shuffle only training data and seed workers when reproducibility matters. Multiple workers can duplicate iterable data, increase memory, and complicate ordering; begin with zero workers. Inspect several decoded batches, including the last short batch, before training. Device transfer usually happens after loading, optionally aided by pinned CPU memory on CUDA.


In [ ]:
from torch.utils.data import Dataset,DataLoader
class Sequences(Dataset):
 def __init__(self,rows): self.rows=rows
 def __len__(self): return len(self.rows)
 def __getitem__(self,i): return torch.tensor(self.rows[i],dtype=torch.long),i%3
def collate(records):
 seqs,targets=zip(*records); length=max(map(len,seqs)); ids=torch.zeros(len(seqs),length,dtype=torch.long); mask=torch.zeros_like(ids,dtype=torch.bool)
 for i,s in enumerate(seqs): ids[i,:len(s)]=s; mask[i,:len(s)]=True
 return {"ids":ids,"mask":mask,"targets":torch.tensor(targets)}
loader=DataLoader(Sequences([[1,2,3],[4],[2,3],[5,6,7,8]]),batch_size=3,shuffle=False,collate_fn=collate)
for batch in loader: print({k:(v.shape,v.dtype) for k,v in batch.items()},batch["ids"])


## 2.9 A correct train/evaluate loop

The core update is: set training mode, clear gradients, run the forward pass, compute a correctly reduced loss, backpropagate, optionally clip, step the optimizer, then step schedulers defined per update. `zero_grad(set_to_none=True)` avoids unnecessary writes and makes missing gradients visible. Gradient accumulation divides each microbatch contribution according to the intended global reduction and steps only after the accumulation boundary; unequal token counts need summed losses and counts rather than a simple division by number of microbatches. Evaluation saves the previous mode, enters inference mode, aggregates numerators and denominators, and restores mode. Log learning rate, loss denominator, gradient norm, and throughput.


In [ ]:
torch.manual_seed(8); model=TinyClassifier(); optimizer=torch.optim.AdamW(model.parameters(),lr=3e-2)
def run_epoch(model,loader,training):
 previous=model.training; model.train(training); total_loss=total_items=0
 context=torch.enable_grad() if training else torch.inference_mode()
 with context:
  for batch in loader:
   if training: optimizer.zero_grad(set_to_none=True)
   logits=model(batch["ids"],batch["mask"]); summed=nn.functional.cross_entropy(logits,batch["targets"],reduction="sum")
   if training:
    (summed/len(batch["targets"])).backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
   total_loss+=summed.item(); total_items+=len(batch["targets"])
 model.train(previous); return total_loss/total_items
for epoch in range(8): train_loss=run_epoch(model,loader,True)
print("train",train_loss,"eval",run_epoch(model,loader,False))


## 2.10 Precision, compilation, and performance measurement

FP32 is the debugging baseline. FP16 saves memory but has a narrow exponent range; BF16 retains FP32-like exponent range and is often preferable on supporting accelerators. Autocast chooses eligible operation dtypes, while parameters may remain FP32. CUDA FP16 commonly uses gradient scaling; unscale before clipping and record skipped steps. Device support differs, so never assume a CUDA optimization applies to MPS or CPU. `torch.compile` can reduce Python and kernel overhead but introduces compilation cost, graph breaks, and shape specialization. Warm up before timing, synchronize accelerators, measure realistic shapes, and check numerical quality after every precision or compiler change.


In [ ]:
print("available:",{"cuda":torch.cuda.is_available(),"mps":torch.backends.mps.is_available()})
for dtype in (torch.float32,torch.float16,torch.bfloat16):
 print(dtype,"bytes/value",torch.empty((),dtype=dtype).element_size(),"range",torch.finfo(dtype).min,torch.finfo(dtype).max)
# Use autocast only for a supported device after establishing an FP32 reference.


## 2.11 Checkpoints, RNG state, and exact resume

A deployment state dictionary is not a resumable training checkpoint. Exact continuation may require model, optimizer, scheduler, scaler, step, sampler or data position, configuration, and random-number-generator states for PyTorch, CUDA, NumPy, and Python. Save ordinary dictionaries rather than whole Python model objects; reconstruct code and load with `weights_only=True` where supported. Move optimizer state to the correct device after loading. Test the artifact in a fresh process and compare an uninterrupted run with a save/resume branch on fixed batches. Determinism also depends on kernels, library versions, hardware, worker order, and distributed collectives, so document the level actually promised.


In [ ]:
from pathlib import Path
checkpoint={"model":model.state_dict(),"optimizer":optimizer.state_dict(),"torch_rng":torch.get_rng_state(),"step":8,"config":{"vocab":16,"width":8,"classes":3}}
path=Path("artifacts/prerequisites/model.pt"); path.parent.mkdir(parents=True,exist_ok=True); torch.save(checkpoint,path)
restored=TinyClassifier(**checkpoint["config"]); loaded=torch.load(path,map_location="cpu",weights_only=True); restored.load_state_dict(loaded["model"]); print("restored step",loaded["step"]); torch.testing.assert_close(model(next(iter(loader))["ids"],next(iter(loader))["mask"]),restored(next(iter(loader))["ids"],next(iter(loader))["mask"]))


## 2.12 A practical debugging ladder

Debug from cheapest invariant to most integrated behavior. First assert shapes, dtypes, ranges, devices, and finite values. Next compare a tensor operation with a hand-worked or NumPy reference and run a gradient check. Inspect a real collated batch and its masks. Overfit one tiny batch with dropout disabled; failure points to the data, forward pass, loss, gradients, optimizer, or modes. Confirm evaluation is deterministic and independent of batch partitioning. Test save/resume equivalence. Only then introduce mixed precision, accumulation, checkpointing, compilation, more workers, or distribution one axis at a time. Preserve every discovered failure as a regression test rather than deleting the evidence.


In [ ]:
def audit(model):
 rows=[]
 for name,p in model.named_parameters():
  rows.append({"name":name,"shape":tuple(p.shape),"trainable":p.requires_grad,"finite":bool(torch.isfinite(p).all()),"grad":None if p.grad is None else float(p.grad.norm())})
 return rows
print(*audit(model),sep=" | ")
assert all(row["finite"] for row in audit(model))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [PyTorch tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
- [PyTorch autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [PyTorch data utilities](https://docs.pytorch.org/docs/stable/data.html)


## Exercises

    1. Implement token-weighted gradient accumulation and prove it matches one large batch.
2. Overfit a padded variable-length batch, then intentionally break and diagnose its mask.
3. Add a scheduler and demonstrate whether it steps per update or per epoch.
4. Prove a dropout-containing model resumes identically after restoring RNG state.
5. Profile a vectorized operation after warm-up on your available accelerator.
6. Write a test that catches an accidental detach or unregistered tensor parameter.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
